<a href="https://colab.research.google.com/github/pushpakgaidhane294/Gen_AI_Lab/blob/main/Practical6_Gen_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Install required dependencies
!pip install -q transformers datasets evaluate rouge_score accelerate ipywidgets

import torch
import evaluate
import ipywidgets as widgets
from IPython.display import display, clear_output
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# 2. Select execution device (GPU if available in Colab)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running on device: {device}")

# 3. Load pre-trained BART model and tokenizer directly
MODEL_NAME = "facebook/bart-large-cnn"
print(f"Loading model {MODEL_NAME}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)

# 4. Define summarization function using model.generate()
def summarize_text(text, max_len=100, min_len=25):
    inputs = tokenizer(text, max_length=1024, truncation=True, return_tensors="pt").to(device)
    summary_ids = model.generate(
        inputs["input_ids"],
        num_beams=4,
        max_length=max_len,
        min_length=min_len,
        length_penalty=2.0,
        early_stopping=True
    )
    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

# 5. Test Sample News Article
sample_article = """
The European Space Agency (ESA) successfully launched its Juice spacecraft on an eight-year voyage to Jupiter's icy moons on Friday.
The Jupiter Icy Moons Explorer (Juice) blasted off on an Ariane 5 rocket from Europe's spaceport in Kourou, French Guiana,
following a 24-hour delay caused by the risk of lightning.

Shortly after launch, ground controllers confirmed telemetry signals and solar array deployment, marking the mission's
first critical milestone. The spacecraft will perform several gravity-assist flybys around Earth and Venus before arriving
at Jupiter in July 2031. Scientists hope to investigate whether Jupiter's moons—Ganymede, Callisto, and Europa—possess
subsurface liquid oceans capable of supporting primitive microbial life. The mission cost is estimated at approximately $1.7 billion.
"""

reference_summary = "ESA launched the $1.7B Juice spacecraft to study Jupiter's icy moons for potential life."

# Generate sample summary & calculate ROUGE
generated_summary = summarize_text(sample_article)
rouge = evaluate.load("rouge")
scores = rouge.compute(predictions=[generated_summary], references=[reference_summary])

# 6. Display Initial Results
print("\n=======================================================")
print("                   PRACTICAL RESULTS                   ")
print("=======================================================")
print(f"\n[ORIGINAL ARTICLE]:\n{sample_article.strip()}")
print(f"\n[GENERATED SUMMARY]:\n{generated_summary}")
print(f"\n[REFERENCE SUMMARY]:\n{reference_summary}")
print("\n[ROUGE EVALUATION METRICS]:")
print(f"  • ROUGE-1 : {scores['rouge1']:.4f}")
print(f"  • ROUGE-2 : {scores['rouge2']:.4f}")
print(f"  • ROUGE-L : {scores['rougeL']:.4f}")
print("=======================================================\n")

# 7. Multi-Line Interactive Interface Box
print("--- MULTI-PARAGRAPH INTERACTIVE SUMMARIZER ---")
print("Paste full multi-paragraph text in the box below and click 'Summarize Article':\n")

text_area = widgets.Textarea(
    value='',
    placeholder='Paste your full multi-line or multi-paragraph news article here...',
    description='Article:',
    disabled=False,
    layout=widgets.Layout(width='95%', height='200px')
)

button = widgets.Button(
    description='Summarize Article',
    button_style='primary',
    icon='magic'
)

output_box = widgets.Output()

def on_button_clicked(b):
    with output_box:
        clear_output()
        user_text = text_area.value.strip()
        if not user_text:
            print("Please paste an article into the text box first.")
            return
        if len(user_text.split()) < 15:
            print("Article is too short. Please enter at least 15 words.")
            return

        print("Generating summary, please wait...\n")
        summary = summarize_text(user_text, max_len=100, min_len=25)
        print("=======================================================")
        print("                 GENERATED SUMMARY                     ")
        print("=======================================================")
        print(summary)
        print("=======================================================")

button.on_click(on_button_clicked)
display(text_area, button, output_box)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 29.2 MB/s eta 0:00:00
Running on device: cuda
Loading model facebook/bart-large-cnn...


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]


                   PRACTICAL RESULTS                   

[ORIGINAL ARTICLE]:
The European Space Agency (ESA) successfully launched its Juice spacecraft on an eight-year voyage to Jupiter's icy moons on Friday. 
The Jupiter Icy Moons Explorer (Juice) blasted off on an Ariane 5 rocket from Europe's spaceport in Kourou, French Guiana, 
following a 24-hour delay caused by the risk of lightning.

Shortly after launch, ground controllers confirmed telemetry signals and solar array deployment, marking the mission's 
first critical milestone. The spacecraft will perform several gravity-assist flybys around Earth and Venus before arriving 
at Jupiter in July 2031. Scientists hope to investigate whether Jupiter's moons—Ganymede, Callisto, and Europa—possess 
subsurface liquid oceans capable of supporting primitive microbial life. The mission cost is estimated at approximately $1.7 billion.

[GENERATED SUMMARY]:
The Jupiter Icy Moons Explorer (Juice) blasted off on an Ariane 5 rocket from Europe

Textarea(value='', description='Article:', layout=Layout(height='200px', width='95%'), placeholder='Paste your…

Button(button_style='primary', description='Summarize Article', icon='magic', style=ButtonStyle())

Output()